In [2]:
import pandas as pd
from pathlib import Path
import ast
import numpy as np

# 1. Load and Normalize Reference Data
# We strip whitespace and convert to lowercase to ensure perfect matching
bigram_counts = pd.read_csv('/home/lillianchang/Documents/MOUS_hierarchical-representations/tables/bigram_counts.csv', index_col=0)
bigram_counts['Bigram'] = bigram_counts['Bigram'].astype(str).str.strip().str.lower()

source = Path('/media/lillianchang/MOUSnew/SynologyDrive/source/SynologyDrive')

for sub in source.iterdir():
    if sub.name.startswith('sub-V'):
        input_file = sub / 'func' / f'{sub.name}_bigrams_processed.csv'
        
        # Only proceed if the input file actually exists
        if input_file.exists():
            try:
                bigrams_processed = pd.read_csv(input_file)
                
                # Create working copy
                bigrams_first = bigrams_processed[['Onset', 'Word', 'Bigrams']].copy()

                # Apply literal_eval to convert string representation of list to actual list
                bigrams_first['Bigrams'] = bigrams_first['Bigrams'].apply(ast.literal_eval)

                # Extract first bigram safely
                bigrams_first['First Bigram'] = bigrams_first['Bigrams'].apply(lambda x: x[0] if len(x) > 0 else None)

                # --- FIX: Normalize the Subject Data before Merging ---
                # Create a temporary key column that is guaranteed to be a lowercase string with no spaces
                bigrams_first['merge_key'] = bigrams_first['First Bigram'].astype(str).str.strip().str.lower()

                # --- FIX: Robust Merge ---
                bigrams_first = bigrams_first.merge(
                    bigram_counts, 
                    left_on='merge_key', 
                    right_on='Bigram', 
                    how='left'
                )

                # --- FIX: Handle Missing Values ---
                # If a bigram isn't in the reference list, 'Count' will be NaN. 
                # We fill NaNs with 0 so the script doesn't crash later.
                if bigrams_first['Count'].isna().any():
                    print(f"Warning: {sub.name} had unmatched bigrams. Filling NaNs with 0.")
                    bigrams_first['Count'] = bigrams_first['Count'].fillna(0)

                # Rename and Calculate Zipf
                bigrams_first = bigrams_first.rename(columns={'Count': 'First_Bigram_Count'})
                bigrams_first['First_bigram_Zipf'] = np.log10(1 + bigrams_first['First_Bigram_Count'])

                # Clean up columns (remove the temp merge key and extra Bigram column)
                cols_to_keep = ['Onset', 'Word', 'Bigrams', 'First Bigram', 'First_Bigram_Count', 'First_bigram_Zipf']
                final_df = bigrams_first[cols_to_keep]

                # Save
                final_df.to_csv(sub / 'func' / f'{sub.name}_first_bigram_zipf.csv')
            
            except Exception as e:
                print(f"Error processing {sub.name}: {e}")

In [3]:
import pandas as pd
from pathlib import Path
import numpy as np

source = Path('/media/lillianchang/MOUSnew/SynologyDrive/source/SynologyDrive')

print(f"{'Subject':<15} | {'NaNs':<10} | {'Std Dev':<10} | {'Status'}")
print("-" * 55)

bad_subjects = []

for sub in source.iterdir():
    if sub.name.startswith('sub-V'):
        target_file = sub / 'func' / f'{sub.name}_first_bigram_zipf.csv'
        
        if target_file.exists():
            df = pd.read_csv(target_file)
            
            # Check 1: Any NaNs in the Zipf column?
            n_nans = df['First_bigram_Zipf'].isna().sum()
            
            # Check 2: Is the variance zero? (Standard Deviation == 0)
            # If std_dev is 0, the column is constant (e.g., all 0s), which crashes SPM contrasts.
            std_dev = df['First_bigram_Zipf'].std()
            
            status = "OK"
            if n_nans > 0:
                status = "FAIL (NaNs)"
                bad_subjects.append(sub.name)
            elif std_dev == 0 or np.isnan(std_dev):
                status = "FAIL (Const)"
                bad_subjects.append(sub.name)
                
            print(f"{sub.name:<15} | {n_nans:<10} | {std_dev:.4f}     | {status}")

print("\n" + "="*30)
if bad_subjects:
    print(f"WARNING: The following subjects will crash SPM:\n{bad_subjects}")
    print("Please inspect their CSVs or exclude them from the 1st level analysis.")
else:
    print("All subjects look valid for SPM analysis.")


Subject         | NaNs       | Std Dev    | Status
-------------------------------------------------------
sub-V1076       | 0          | 0.4708     | OK
sub-V1093       | 0          | 0.4958     | OK
sub-V1009       | 0          | 0.4704     | OK
sub-V1028       | 0          | 0.4708     | OK
sub-V1084       | 0          | 0.4958     | OK
sub-V1058       | 0          | 0.4708     | OK
sub-V1104       | 0          | 0.4911     | OK
sub-V1087       | 0          | 0.4958     | OK
sub-V1045       | 0          | 0.4958     | OK
sub-V1106       | 0          | 0.4708     | OK
sub-V1111       | 0          | 0.4958     | OK
sub-V1039       | 0          | 0.4958     | OK
sub-V1108       | 0          | 0.4958     | OK
sub-V1052       | 0          | 0.4708     | OK
sub-V1098       | 0          | 0.4911     | OK
sub-V1117       | 0          | 0.4958     | OK
sub-V1110       | 0          | 0.4911     | OK
sub-V1044       | 0          | 0.4911     | OK
sub-V1048       | 0          | 0.4958     | OK


In [4]:
import pandas as pd
from pathlib import Path
import ast

# 1. Load Reference Data (The "Dictionary")
print("Loading reference data...")
bigram_counts = pd.read_csv('/home/lillianchang/Documents/MOUS_hierarchical-representations/tables/bigram_counts.csv', index_col=0)
# Create a set of valid keys for fast lookup
valid_bigrams = set(bigram_counts['Bigram'].astype(str).str.strip().str.lower())

# 2. Setup Source Directory
source = Path('/media/lillianchang/MOUSnew/SynologyDrive/source/SynologyDrive')
unmatched_data = []

print("Scanning subjects for mismatches...")

# 3. Iterate and Check
for sub in source.iterdir():
    if sub.name.startswith('sub-V'):
        input_file = sub / 'func' / f'{sub.name}_bigrams_processed.csv'
        
        if input_file.exists():
            try:
                # Load subject data
                df = pd.read_csv(input_file)
                
                # Convert string representation of list to actual list
                df['Bigrams'] = df['Bigrams'].apply(ast.literal_eval)
                
                # Get the first bigram (ignoring empty lists for now)
                df['First_Bigram_Raw'] = df['Bigrams'].apply(lambda x: x[0] if len(x) > 0 else None)
                
                # Filter out rows where there IS a bigram but it didn't match
                # (We ignore rows where bigram is None, as those are just empty)
                df_check = df[df['First_Bigram_Raw'].notna()].copy()
                
                # Create the clean key we used for matching
                df_check['Clean_Key'] = df_check['First_Bigram_Raw'].astype(str).str.strip().str.lower()
                
                # Find rows where the key is NOT in our valid set
                mismatches = df_check[~df_check['Clean_Key'].isin(valid_bigrams)]
                
                # Record findings
                for _, row in mismatches.iterrows():
                    unmatched_data.append({
                        'Bigram_Used': row['Clean_Key'],
                        'Original_Word': row['Word'],
                        'Subject': sub.name
                    })
                    
            except Exception as e:
                print(f"Could not read {sub.name}: {e}")

# 4. Print the Report
if unmatched_data:
    report_df = pd.DataFrame(unmatched_data)
    
    print(f"\n{'='*20} MISSING BIGRAM REPORT {'='*20}")
    print(f"Total unmatched instances found: {len(report_df)}")
    
    # Group to show the most common offenders
    summary = report_df.groupby('Bigram_Used').agg(
        Total_Failures=('Subject', 'count'),
        Example_Words=('Original_Word', lambda x: list(set(x))[:3]) # Show up to 3 unique examples
    ).sort_values('Total_Failures', ascending=False)
    
    print("\nTop missing bigrams (Check these against your reference file):")
    print(summary.head(15))
    
    print(f"\n{'='*60}")
else:
    print("\nNo mismatches found! (If you saw warnings before, check if bigram_counts.csv was loaded correctly)")

Loading reference data...
Scanning subjects for mismatches...

No mismatches found! (If you saw warnings before, check if bigram_counts.csv was loaded correctly)
